# 带反馈回路的检索（Retrieval with Feedback Loop）

标准 RAG 往往是“一次检索 → 一次生成”。现实里会遇到两个常见问题：

- **检索不稳定**：同一主题下，不同问法会导致检索命中文档不同，答案质量波动。
- **系统不记错/不记好**：用户刚刚指出“这个回答不相关/很好”，下一次系统仍可能重复同样的问题。

这一节我们做一个简化但可运行的 **Feedback Loop RAG**：

- 把用户对回答的 **相关性/质量** 反馈持久化（JSONL）
- 用历史高质量反馈来 **改写查询**、并在候选文档里做 **轻量重排**
- 定期把“高质量问答”写回索引，形成 **知识库的增量强化**


<div style="text-align: center;">

<img src="../images/retrieval_with_feedback_loop.svg" alt="retrieval with feedback loop" style="width:45%; height:auto;">
</div>

## 0) 准备：环境与依赖

你需要在环境中设置 `OPENAI_API_KEY`（可放在项目根目录的 `.env` 里）。本 notebook 使用 LangChain 的 **runnable** 风格，并统一用 `.invoke()` 调用。

In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
from datetime import datetime
from pathlib import Path
from typing import Any, Iterable, Optional

import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pypdf import PdfReader

load_dotenv("../.env")
sys.path.insert(0, str(Path("..").resolve()))

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = DATA_DIR / "Understanding_Climate_Change.pdf"
FEEDBACK_PATH = DATA_DIR / "feedback_data.jsonl"

VSTORE_DIR = Path("../vector_stores")
VSTORE_DIR.mkdir(parents=True, exist_ok=True)

BASE_STORE_DIR = VSTORE_DIR / "feedback_loop_base_faiss"
TUNED_STORE_DIR = VSTORE_DIR / "feedback_loop_tuned_faiss"

/tmp/ipykernel_807882/791998684.py:25: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 1) 准备数据：下载 PDF（如果本地不存在）

In [ ]:
os.environ["HTTP_PROXY"] = os.getenv("HTTP_PROXY", "http://127.0.0.1:7890")
os.environ["HTTPS_PROXY"] = os.getenv("HTTPS_PROXY", "http://127.0.0.1:7890")
os.environ["ALL_PROXY"] = os.getenv("ALL_PROXY", "http://127.0.0.1:7890")

In [3]:
PDF_URL = "https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf"


def download_file(url: str, path: Path, timeout_s: int = 60) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size > 0:
        return
    headers = {"User-Agent": "paper-qa-rag-techniques/1.0"}
    r = requests.get(url, headers=headers, timeout=timeout_s)
    r.raise_for_status()
    path.write_bytes(r.content)


download_file(PDF_URL, PDF_PATH)
PDF_PATH

PosixPath('../data/Understanding_Climate_Change.pdf')

## 2) 从 PDF 抽取文本，并切分为 chunks

In [4]:
def read_pdf_pages(path: Path) -> list[Document]:
    reader = PdfReader(str(path))
    docs: list[Document] = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = re.sub(r"\s+", " ", text).strip()
        if not text:
            continue
        docs.append(
            Document(
                page_content=text,
                metadata={"source": str(path), "page": i, "type": "PDF_PAGE_TEXT"},
            )
        )
    return docs


pages = read_pdf_pages(PDF_PATH)
len(pages), pages[0].metadata

(33,
 {'source': '../data/Understanding_Climate_Change.pdf',
  'page': 0,
  'type': 'PDF_PAGE_TEXT'})

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(pages)
for d in chunks:
    d.metadata["type"] = "PDF_TEXT_CHUNK"

len(chunks), chunks[0].metadata

(97,
 {'source': '../data/Understanding_Climate_Change.pdf',
  'page': 0,
  'type': 'PDF_TEXT_CHUNK'})

## 3) 建立（或加载）基础向量索引

我们先做一个“基础索引”（只包含 PDF chunks）。后面会演示如何把高质量反馈写回索引，得到“tuned 索引”。

In [6]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


def load_or_build_base_store() -> FAISS:
    if BASE_STORE_DIR.exists():
        return FAISS.load_local(
            str(BASE_STORE_DIR),
            embeddings,
            allow_dangerous_deserialization=True,
        )
    store = FAISS.from_documents(chunks, embedding=embeddings)
    store.save_local(str(BASE_STORE_DIR))
    return store


base_store = load_or_build_base_store()
base_retriever = base_store.as_retriever(search_kwargs={"k": 6})

BASE_STORE_DIR

PosixPath('../vector_stores/feedback_loop_base_faiss')

## 4) 基线 RAG：一次检索 → 一次生成

这里我们用一个最小可复用的 runnable chain：

- `retriever.invoke(question)` 取回 `Document[]`
- 把文档拼成 `context`
- 用 prompt + LLM 生成答案


In [7]:
def docs_to_context(docs: list[Document]) -> str:
    lines: list[str] = []
    for doc in docs:
        page = doc.metadata.get("page", "?")
        lines.append(f"[page={page}] {doc.page_content}")
    return "\n\n".join(lines)


qa_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个严谨的问答助手。只根据给定的检索上下文回答；如果上下文不足以支撑结论，就明确说不知道。",
        ),
        (
            "human",
            "问题：{question}\n\n检索上下文：\n{context}\n\n请给出中文回答：",
        ),
    ]
)

rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": base_retriever | RunnableLambda(docs_to_context),
    }
    | qa_prompt
    | qa_llm
    | StrOutputParser()
)

question = "什么是温室效应（greenhouse effect）？"
answer = rag_chain.invoke(question)
answer[:600]

'温室效应是指大气中温室气体（如二氧化碳、甲烷和氧化亚氮）吸收来自太阳的热量，从而使地球保持足够温暖以支持生命的过程。虽然这一自然过程对地球生命至关重要，但人类活动的加剧，特别是燃烧化石燃料，导致温室气体浓度增加，从而加剧了温室效应，导致气候变暖。'

## 5) 反馈数据：存储与读取（JSONL）

我们把每次交互的反馈作为一行 JSON 存下来（方便追加、也方便离线分析）。这里的 `relevance/quality` 用 1-5 的整数刻度。

In [8]:
class FeedbackRecord(BaseModel):
    query: str
    response: str
    relevance: int = Field(..., ge=1, le=5)
    quality: int = Field(..., ge=1, le=5)
    comments: str = ""
    created_at: str = Field(default_factory=lambda: datetime.utcnow().isoformat())


def append_feedback(record: FeedbackRecord, path: Path = FEEDBACK_PATH) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(record.model_dump_json(ensure_ascii=False) + "\n")


def load_feedback(path: Path = FEEDBACK_PATH, limit: Optional[int] = None) -> list[FeedbackRecord]:
    if not path.exists():
        return []
    items: list[FeedbackRecord] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            items.append(FeedbackRecord.model_validate_json(line))
            if limit is not None and len(items) >= limit:
                break
    return items


feedback_items = load_feedback()
len(feedback_items), (feedback_items[-1].model_dump() if feedback_items else None)

(1,
 {'query': '什么是温室效应（greenhouse effect）？',
  'response': '温室效应是指大气中的温室气体（如二氧化碳、甲烷和氧化亚氮）捕获来自太阳的热量，从而使地球保持足够温暖以支持生命的过程。这一自然现象对地球生命至关重要。然而，人类活动加剧了这一过程，导致气候变暖。',
  'relevance': 5,
  'quality': 5,
  'comments': '回答清晰，覆盖了关键机制',
  'created_at': '2026-06-19T07:51:42.580425'})

## 6) 用历史反馈改进下一次检索

我们做两件事（都尽量“轻量”，避免为每个 doc 单独跑一遍 LLM）：

1. **挑选与当前问题相关的高质量反馈**（用一个小型向量索引做近邻检索）
2. 让 LLM 生成：
   - 一个更适合检索的 **改写查询**
   - 一组 **focus terms**（关键词/关键短语），用于对候选文档做简单重排


In [9]:
def build_feedback_store(items: list[FeedbackRecord]) -> Optional[FAISS]:
    good = [x for x in items if x.relevance >= 4 and x.quality >= 4]
    if not good:
        return None
    docs = [
        Document(
            page_content=f"Q: {x.query}\nA: {x.response}",
            metadata={
                "type": "FEEDBACK_QA",
                "relevance": x.relevance,
                "quality": x.quality,
                "created_at": x.created_at,
            },
        )
        for x in good
    ]
    return FAISS.from_documents(docs, embedding=embeddings)


feedback_store = build_feedback_store(load_feedback())

In [10]:
class RewriteOutput(BaseModel):
    rewritten_query: str = Field(..., description="更适合检索的改写问题")
    focus_terms: list[str] = Field(default_factory=list, description="用于重排的关键词/关键短语")
    use_feedback: bool = Field(..., description="是否真正用到了反馈信息")


rewrite_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个检索查询优化器。给你用户问题与历史高质量反馈片段。"
            "目标：产生一个更利于向量检索的改写问题，并给出少量用于重排的关键词/短语。"
            "不要编造反馈里没有的信息。focus_terms 最多 8 个。",
        ),
        (
            "human",
            "用户问题：{question}\n\n历史反馈片段（可能为空）：\n{feedback_snippets}",
        ),
    ]
)

rewrite_chain = rewrite_prompt | rewrite_llm.with_structured_output(
    RewriteOutput, method="json_schema", strict=True
)


def format_feedback_snippets(docs: list[Document]) -> str:
    if not docs:
        return "(无)"
    out = []
    for i, d in enumerate(docs, start=1):
        out.append(f"[{i}] {d.page_content}")
    return "\n\n".join(out)


def get_relevant_feedback(question: str, k: int = 3) -> list[Document]:
    if feedback_store is None:
        return []
    return feedback_store.similarity_search(question, k=k)


test_q = "解释一下温室气体如何导致温室效应"
snippets = get_relevant_feedback(test_q, k=3)
rewrite_chain.invoke({"question": test_q, "feedback_snippets": format_feedback_snippets(snippets)})

RewriteOutput(rewritten_query='温室气体如何影响温室效应的机制是什么？', focus_terms=['温室气体', '温室效应', '机制', '气候变暖', '二氧化碳', '甲烷', '氧化亚氮', '人类活动'], use_feedback=True)

### 6.1 反馈感知的检索（候选集 + 关键词轻量重排）

对向量检索最稳妥的做法之一是：

- 先用（改写后的）query 在向量索引里 **取一个较大的候选集**（例如 20）
- 再用一个很便宜的规则（这里是 focus_terms 的词面覆盖）做 **轻量重排**

这样可以把“反馈带来的偏好/纠错信号”注入到结果里，但不会让每次检索都变成大量 LLM 调用。

In [11]:
def keyword_overlap_score(text: str, terms: Iterable[str]) -> float:
    t = text.lower()
    return float(sum(1 for term in terms if term and term.lower() in t))


def feedback_aware_search(
    store: FAISS,
    question: str,
    fetch_k: int = 20,
    final_k: int = 6,
    boost: float = 0.35,
) -> tuple[list[Document], RewriteOutput]:
    fb_docs = get_relevant_feedback(question, k=3)
    rewrite = rewrite_chain.invoke(
        {"question": question, "feedback_snippets": format_feedback_snippets(fb_docs)}
    )
    q = (rewrite.rewritten_query or question).strip()

    # FAISS: similarity_search_with_score 返回的是 (doc, score)
    # 对于 FAISS 这里通常是 L2 distance，越小越相似。
    candidates = store.similarity_search_with_score(q, k=fetch_k)

    reranked: list[tuple[float, Document]] = []
    for doc, dist in candidates:
        overlap = keyword_overlap_score(doc.page_content, rewrite.focus_terms)
        combined = (-float(dist)) + boost * overlap
        doc.metadata = {**doc.metadata, "_dist": float(dist), "_overlap": overlap, "_combined": combined}
        reranked.append((combined, doc))
    reranked.sort(key=lambda x: x[0], reverse=True)
    return [d for _, d in reranked[:final_k]], rewrite


def show_docs(docs: list[Document], max_chars: int = 220) -> None:
    for i, d in enumerate(docs, start=1):
        page = d.metadata.get("page", "?")
        dist = d.metadata.get("_dist", None)
        overlap = d.metadata.get("_overlap", None)
        combined = d.metadata.get("_combined", None)
        head = d.page_content[:max_chars].replace("\n", " ")
        print(f"[{i}] page={page} dist={dist} overlap={overlap} combined={combined}")
        print(head)
        print("-")


docs_fb, rewrite = feedback_aware_search(base_store, "什么是温室效应？")
rewrite, len(docs_fb)

(RewriteOutput(rewritten_query='温室效应的定义及其对气候的影响是什么？', focus_terms=['温室效应', '定义', '气候影响', '温室气体', '二氧化碳', '甲烷', '气候变暖', '人类活动'], use_feedback=True),
 6)

In [12]:
show_docs(docs_fb)

[1] page=2 dist=1.09395170211792 overlap=0.0 combined=-1.09395170211792
carbon footprint. Chapter 3: Effects of Climate Change The effects of climate change are already being felt around the world and are projected to intensify in the coming decades. These effects include: Rising Temperature
-
[2] page=0 dist=1.178797721862793 overlap=0.0 combined=-1.178797721862793
is the increase in greenhouse gases in the atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essent
-
[3] page=0 dist=1.2329744100570679 overlap=0.0 combined=-1.2329744100570679
Understanding Climate Change Chapter 1: Introduction to Climate Change Climate change refers to significant, long-term changes in the global climate. The term "global climate" encompasses the planet's overall weather pat
-
[4] page=0 dist=1.2727129459381104 overlap=0.0 combined=-1.2727129459381104
that change the amount of solar e

## 7) 演示：先产生一次反馈，再让它影响下一次检索

为了让 notebook 可重复运行且不需要交互输入，我们用“模拟反馈”的方式演示：

- 第一次问：得到回答 → 写入一条高分反馈
- 第二次问（换一种问法）：比较 **基线** vs **反馈感知检索** 的结果差异


In [13]:
q1 = "什么是温室效应（greenhouse effect）？"
a1 = rag_chain.invoke(q1)

# 这里模拟用户觉得回答很好
append_feedback(
    FeedbackRecord(
        query=q1,
        response=a1,
        relevance=5,
        quality=5,
        comments="回答清晰，覆盖了关键机制",
    )
)

len(load_feedback())

/tmp/ipykernel_807882/1041531682.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at: str = Field(default_factory=lambda: datetime.utcnow().isoformat())


2

In [14]:
# 重新构建 feedback_store（因为 feedback 文件刚刚新增了一条）
feedback_store = build_feedback_store(load_feedback())

q2 = "温室气体是怎么导致地球变暖的？（请解释温室效应机制）"

baseline_docs = base_retriever.invoke(q2)
baseline_answer = rag_chain.invoke(q2)

fb_docs, rewrite2 = feedback_aware_search(base_store, q2)
fb_context = docs_to_context(fb_docs)
fb_answer = (
    {
        "question": RunnablePassthrough(),
        "context": RunnableLambda(lambda _: fb_context),
    }
    | qa_prompt
    | qa_llm
    | StrOutputParser()
).invoke(q2)

print("改写查询：", rewrite2.rewritten_query)
print("focus_terms：", rewrite2.focus_terms)

print("\n--- 基线检索 top-3 ---")
show_docs(baseline_docs[:3])

print("\n--- 反馈感知检索 top-3 ---")
show_docs(fb_docs[:3])

print("\n--- 基线回答（截断） ---\n", baseline_answer[:500])
print("\n--- 反馈感知回答（截断） ---\n", fb_answer[:500])

改写查询： 温室气体如何通过温室效应导致地球变暖？请详细解释其机制。
focus_terms： ['温室气体', '温室效应', '地球变暖', '机制', '二氧化碳', '甲烷', '氧化亚氮', '人类活动']

--- 基线检索 top-3 ---
[1] page=0 dist=1.1034207344055176 overlap=0.0 combined=-1.1034207344055176
is the increase in greenhouse gases in the atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essent
-
[2] page=0 dist=1.1941479444503784 overlap=0.0 combined=-1.1941479444503784
that change the amount of solar energy our planet receives. During the Holocene epoch, which began at the end of the last ice age, human societies flourished, but the industrial era has seen unprecedented changes. Modern
-
[3] page=2 dist=1.202555537223816 overlap=0.0 combined=-1.202555537223816
carbon footprint. Chapter 3: Effects of Climate Change The effects of climate change are already being felt around the world and are projected to intensify in the coming decades. These effects includ

## 8) 周期性索引微调：把高质量问答写回向量库

原理很直接：

- 当某些问答被用户连续打高分时，它们本质上是“被验证过的知识片段”。
- 我们可以把 `Q + A` 作为一类新文档加入向量库（metadata 标记为 `FEEDBACK_DOC`），以后相似问题更容易检索到这些片段。

这不是唯一做法，但它非常容易落地，也能直观展示“反馈 → 检索”闭环。

In [15]:
def fine_tune_store_from_feedback(
    base: FAISS,
    feedback: list[FeedbackRecord],
    tuned_dir: Path = TUNED_STORE_DIR,
) -> FAISS:
    good = [x for x in feedback if x.relevance >= 4 and x.quality >= 4]
    if not good:
        return base

    docs = [
        Document(
            page_content=(
                "这是一条被用户高分认可的问答对。\n"
                f"用户问题：{x.query}\n"
                f"系统回答：{x.response}"
            ),
            metadata={
                "type": "FEEDBACK_DOC",
                "relevance": x.relevance,
                "quality": x.quality,
                "created_at": x.created_at,
            },
        )
        for x in good
    ]

    # 复制一份索引再增量写入（避免覆盖 base）
    tuned = base
    tuned.add_documents(docs)
    tuned_dir.mkdir(parents=True, exist_ok=True)
    tuned.save_local(str(tuned_dir))
    return tuned


tuned_store = fine_tune_store_from_feedback(base_store, load_feedback())
tuned_retriever = tuned_store.as_retriever(search_kwargs={"k": 6})
TUNED_STORE_DIR

PosixPath('../vector_stores/feedback_loop_tuned_faiss')

In [16]:
tuned_chain = (
    {
        "question": RunnablePassthrough(),
        "context": tuned_retriever | RunnableLambda(docs_to_context),
    }
    | qa_prompt
    | qa_llm
    | StrOutputParser()
)

q3 = "用一句话解释温室效应的机制"
print("tuned answer:\n", tuned_chain.invoke(q3)[:500])

tuned answer:
 温室效应的机制是大气中的温室气体吸收和保留来自太阳的热量，从而使地球保持温暖。
